# The "Teacher-Student" Gap in Foundation Vision Models



In [1]:
! pip install torch

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
print(f'PyTorch version: {torch.__version__}')

PyTorch version: 2.10.0+cpu


---
## Knowledge Distillation

The combined distillation loss:

$$\mathcal{L} = (1 - \alpha)\mathcal{L}_{CE}(y, \hat{p}) + \alpha\,\tau^2\,\mathcal{L}_{CE}(q^T, q^S)$$

Minimising $\mathcal{L}_{CE}(q^T, q^S)$ over student parameters $\equiv$ minimising $\text{KL}(T \| S) + H(T)$

In [9]:
def hinton_distillation_loss(
    student_logits,
    teacher_logits,
    hard_labels,
    temperature: float = 3.0,
    alpha: float = 0.7
):
    hard_loss = F.cross_entropy(student_logits, hard_labels)

    teacher_soft = F.softmax(teacher_logits / temperature, dim=-1)
    student_log_soft = F.log_softmax(student_logits / temperature, dim=-1)

    soft_loss = -(teacher_soft * student_log_soft).sum(dim=-1).mean()
    soft_loss = soft_loss * (temperature ** 2)

    total_loss = (1 - alpha) * hard_loss + alpha * soft_loss
    return total_loss, hard_loss, soft_loss

---
## KL Divergence: Forward, Reverse, and Jensen-Shannon

| Direction | Formula       | Behaviour |
|-----------|---------------|----------|
| Forward KL | $\text{KL}(T\|S)$ | mean-seeking, blurry attention |
| Reverse KL | $\text{KL}(S\|T)$ | mode-seeking, sharp / collapse risk |
| JSD | $\frac{1}{2}\text{KL}(P\|M)+\frac{1}{2}\text{KL}(Q\|M)$ | symmetric diagnostic |

In [10]:
def forward_kl_loss(student_logits, teacher_logits, temperature: float = 1.0):
    teacher_probs = F.softmax(teacher_logits / temperature, dim=-1)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=-1)

    loss = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean')
    return loss * (temperature ** 2)


def reverse_kl_loss(student_logits, teacher_logits, temperature: float = 1.0):
    student_probs = F.softmax(student_logits / temperature, dim=-1)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=-1)
    teacher_log_probs = F.log_softmax(teacher_logits / temperature, dim=-1)

    kl_per_sample = torch.sum(
        student_probs * (student_log_probs - teacher_log_probs), dim=-1
    )
    return kl_per_sample.mean() * (temperature ** 2)


def jsd_loss(student_logits, teacher_logits, temperature: float = 1.0):
    teacher_probs = F.softmax(teacher_logits / temperature, dim=-1)
    student_probs = F.softmax(student_logits / temperature, dim=-1)

    M = 0.5 * (teacher_probs + student_probs)
    log_M = torch.log(M + 1e-9)

    kl_T_M = torch.sum(teacher_probs * (torch.log(teacher_probs + 1e-9) - log_M), dim=-1)
    kl_S_M = torch.sum(student_probs * (torch.log(student_probs + 1e-9) - log_M), dim=-1)

    jsd = 0.5 * kl_T_M + 0.5 * kl_S_M
    return jsd.mean()

---
## Feature-Space Distillation: ℓ₂ Regression (FitNets)

$$\mathcal{L}_{feat} = \|f_T(x) - W f_S(x)\|_2^2$$

This ℓ₂ loss is equivalent to Forward KL under a Gaussian noise assumption on the residual — another implicit mean-seeking objective.

In [11]:
class FeatureDistillationLoss(nn.Module):
    def __init__(self, student_dim: int, teacher_dim: int):
        super().__init__()
        self.projection = nn.Linear(student_dim, teacher_dim, bias=False)

    def forward(self, student_features, teacher_features):
        aligned_student = self.projection(student_features)
        loss = F.mse_loss(aligned_student, teacher_features.detach())
        return loss

---
## Attention Map Analysis: Entropy & Cosine Similarity

**Row entropy** for token $i$:
$$H(A_i) = -\sum_{j=1}^{N} A_{ij} \log A_{ij}$$

**Mean entropy** over all heads and tokens:
$$\bar{H} = \frac{1}{L \cdot N} \sum_{\ell=1}^{L} \sum_{i=1}^{N} H(A_i^\ell)$$

- High $\bar{H}$ → diffuse attention → Forward KL signature
- Low $\bar{H}$ → sharp/collapsed attention → Reverse KL signature

**Cosine similarity** measures geometric fidelity independent of sharpness:
$$\text{sim}(A^T, A^S) = \frac{\langle A^T, A^S \rangle}{\|A^T\| \cdot \|A^S\|}$$

In [12]:
def calculate_attention_entropy(attention_weights):
    eps = 1e-9
    entropy = -torch.sum(attention_weights * torch.log(attention_weights + eps), dim=-1)
    return entropy.mean()


def attention_cosine_similarity(teacher_attn, student_attn):
    T_flat = teacher_attn.flatten(start_dim=-2)
    S_flat = student_attn.flatten(start_dim=-2)

    cos_sim = F.cosine_similarity(T_flat, S_flat, dim=-1)
    return cos_sim.mean()


def detect_mode_collapse(attention_weights, collapse_threshold: float = 0.5):
    max_weights = attention_weights.max(dim=-1).values   # (...)
    collapse_rate = (max_weights > collapse_threshold).float().mean()
    return collapse_rate.item(), max_weights.mean().item()

---
## Importance-Weighted Reverse KL Estimator

When the teacher is an implicit distribution (defined only through features, not an explicit density), we estimate $\text{KL}(S\|T)$ via importance weighting:

$$\text{KL}(S\|T) = \mathbb{E}_{x \sim q}\left[\frac{S(x)}{q(x)}\log\frac{S(x)}{T(x)}\right]$$

Setting $q = S$ recovers the direct estimator.

In [7]:
def importance_weighted_reverse_kl(
    student_logits,
    teacher_logits,
    proposal_logits=None,
    temperature: float = 1.0
):
    student_probs = F.softmax(student_logits / temperature, dim=-1)
    student_log_probs = F.log_softmax(student_logits / temperature, dim=-1)
    teacher_log_probs = F.log_softmax(teacher_logits / temperature, dim=-1)

    if proposal_logits is None:
        importance_weights = torch.ones(student_logits.shape[0], device=student_logits.device)
    else:
        proposal_probs = F.softmax(proposal_logits / temperature, dim=-1)
        log_ratio = (student_log_probs - torch.log(proposal_probs + 1e-9))
        importance_weights = torch.exp(log_ratio.sum(dim=-1))
        importance_weights = importance_weights.clamp(0.05, 20.0)
    kl_per_class = student_probs * (student_log_probs - teacher_log_probs)
    kl_per_sample = kl_per_class.sum(dim=-1)
    iw_kl = (importance_weights * kl_per_sample).mean()
    return iw_kl * (temperature ** 2), importance_weights.mean().item()

---
## Verification

In [13]:
B, C = 4, 10
N_heads, N_tok = 6, 32
D_teacher, D_student = 384, 128
TAU = 3.0

teacher_logits = torch.randn(B, C)
student_logits = torch.randn(B, C)
hard_labels    = torch.randint(0, C, (B,))

teacher_attn = F.softmax(torch.randn(B, N_heads, N_tok, N_tok), dim=-1)
student_attn = F.softmax(torch.randn(B, N_heads, N_tok, N_tok), dim=-1)

teacher_feats = torch.randn(B, D_teacher)
student_feats = torch.randn(B, D_student)

total_loss, hard_loss, soft_loss = hinton_distillation_loss(
    student_logits, teacher_logits, hard_labels, temperature=TAU, alpha=0.7
)
fwd_loss  = forward_kl_loss(student_logits, teacher_logits, temperature=TAU)
rev_loss  = reverse_kl_loss(student_logits, teacher_logits, temperature=TAU)
jsd       = jsd_loss(student_logits, teacher_logits, temperature=TAU)

feat_distill = FeatureDistillationLoss(student_dim=D_student, teacher_dim=D_teacher)
feat_loss = feat_distill(student_feats, teacher_feats)

teacher_entropy = calculate_attention_entropy(teacher_attn)
student_entropy = calculate_attention_entropy(student_attn)
cos_sim = attention_cosine_similarity(teacher_attn, student_attn)
collapse_rate, mean_max = detect_mode_collapse(student_attn)

iw_kl, mean_w = importance_weighted_reverse_kl(
    student_logits, teacher_logits, temperature=TAU
)


print(f'Hinton total loss: {total_loss.item():.4f}')
print(f'Hard-label CE: {hard_loss.item():.4f}')
print(f'Soft-label: {soft_loss.item():.4f}')
print(f'Forward KL: {fwd_loss.item():.4f}')
print(f'Reverse KL: {rev_loss.item():.4f}')
print(f'Jensen-Shannon divergence: {jsd.item():.4f}')

print(f'Feature ℓ₂ loss: {feat_loss.item():.4f}')

print(f'Teacher attention entropy: {teacher_entropy.item():.4f}')
print(f'Student attention entropy: {student_entropy.item():.4f}')
print(f'Entropy gap: {(student_entropy - teacher_entropy).item():+.4f}')
print(f'Cosine similarity: {cos_sim.item():.4f}')
print(f'Collapse rate: {collapse_rate:.2%}')
print(f'Mean max attention weight: {mean_max:.4f}')


print(f'IW Reverse KL: {iw_kl.item():.4f}')
print(f'Mean importance weight: {mean_w:.4f}')

Hinton total loss: 15.5023
Hard-label CE: 2.6582
Soft-label: 21.0069
Forward KL: 0.5977
Reverse KL: 0.5959
Jensen-Shannon divergence: 0.0164
Feature ℓ₂ loss: 1.4051
Teacher attention entropy: 3.0139
Student attention entropy: 3.0122
Entropy gap: -0.0017
Cosine similarity: 0.4290
Collapse rate: 0.39%
Mean max attention weight: 0.1673
IW Reverse KL: 0.5959
Mean importance weight: 1.0000
